In [ ]:
!pip install rapidfuzz python-docx pytesseract opencv-python Pillow pandas easyocr pdfplumber pdf2image json_repair--quiet

In [125]:
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import pytesseract
import easyocr
import re
from dataclasses import dataclass, field
from typing import List, Optional
import os
import json

from os.path import join
import random
from tqdm.auto import tqdm
import json
import json_repair

from glob import glob
from pdf2image import convert_from_path
from PIL import Image
from PIL import ImageEnhance
from IPython.display import Image as IPImage, display
import easyocr
from rapidfuzz import process


c:\Users\Abdelrahman\anaconda3\envs\New_test\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'json_repair'

In [126]:


doc = Document("med.docx")

names = []

for table in doc.tables:
    for row in table.rows:
        text = row.cells[0].text.strip()
        
        if text and text.lower() != "name":
            names.append(text)

cleaned = []

for name in names:
    name = name.replace("\n", " ")
    name = name.replace('"', '“').replace("'", "“")
    name = re.sub(r"\s+", " ", name).strip()
    
    cleaned.append(name)

cleaned = list(dict.fromkeys(cleaned))
# for i in range(len(cleaned)):
#     print(f"{i+1}. {cleaned[i]}")

names = cleaned

os.makedirs("images", exist_ok=True)

image_paths = []
img_index = 0

for rel in doc.part._rels:
    rel = doc.part._rels[rel]
    if "image" in rel.target_ref:
        img_data = rel.target_part.blob
        
        img_name = f"images/img_{img_index}.png"
        
        with open(img_name, "wb") as f:
            f.write(img_data)
        
        image_paths.append(img_name)
        img_index += 1

# -----------------------
data = []

for i in range(min(len(names), len(image_paths))):
    data.append({
        "name": names[i],
        "image": image_paths[i]
    })

# -----------------------
# 4. عرض النتيجة
# -----------------------
for row in names[:-10]:
    print(row)

Abilify 10-15-5
Abilia -15-20 “abIlIfy”
Acitretin 10-25
Actos 15-30
Acyclovir 400-800
Adancor 10-20
Adwitine 30-60 “cymbalta”
Agovald 25
anarcol
Anastrazol Sandoz “arImIdex”
Anastrazol synthon “arImIdex”
Andovimpamid e 50-100-200 “lacovImp”
Andorivaban 2.5-10-15-20 “xarelto”
Angiobradine 5-7.5 “procoralaN”
Anselacox 60-90 “arcoxIa”
Anvizram 200
Apexidon 0.5-1-2-3-4 Syrup “rIsperdal”
Apixatrack 2.5-5 “elIqUIs”
Apetoid 20 “avara”
Artixiban 2.5-5 “elIqUIs”
Aromasin 25
Aricept 5-10
Arimidex 1
Aripiprex 10-30-vial “abIlIfy”
Arcoxia 60-90
Atomafutix 10-25-40 “strattera ”
Atomox apex 10-18-25-40-60 “strattera ”
Attensera 18 25 40 “strattera ”
Arthfree 20 “avara”
Atoreza 10l10 20l10 40l10 “atozet “
Atozet 10l10 20l10 40l10
Atozemb 10l10 20l10 “atozet “
Avodart 0.5
Avipravir 200
Avara 10-20
Betmiga 50
Bladogra 50 “betmIga”
Brilique 60-90
Bradipect 5-7.5 “procoralaN ”
Brintellix 10-20
Budexan inh 1-0.5
Casodex 50
Carcemia 100 400
Carfalone 25-50
Cardura 1-4-dxl
Celebrex 200-100
Cellcept 250- 500

In [ ]:
from rapidfuzz import process

# database بتاعتك (الأدوية الصح)
drug_db = [
    "Apixaban",
    "Anarcol",
    "Abilify",
    "Cymbalta",
    "Eliquis",
    "Xarelto"
]

def extract_drug(text):
    # خد كل الكلمات
    words = text.split()
    
    best_match = None
    best_score = 0
    
    for w in words:
        match, score, _ = process.extractOne(w, drug_db)
        
        if score > best_score:
            best_match = match
            best_score = score
    
    return best_match

def pipeline(text):
    
    return extract_drug(text)

print(pipeline("IKSARONT Alify Film Cwuted Tahlets"))

In [ ]:
reader = easyocr.Reader(['en'])

names_db = names   # data اللي عملته قبل كده

# =========================
# 3. extract text من الصورة
# =========================
def extract_text(img_path):
    img = cv2.imread(img_path)
    result = reader.readtext(img)
    text = " ".join([r[1] for r in result])
    return text

# =========================
# 4. clean text
# =========================
def clean_text(text):
    text = text.lower()
    return text

# =========================
# 5. match مع names
# =========================
def match_name(text):
    match, score, _ = process.extractOne(text, names_db)
    return match, score

# =========================
# 6. pipeline
# =========================
final_results = []

for item in data:
    img_path = item["image"]
    
    text = extract_text(img_path)
    text = clean_text(text)
    
    matched_name, score = match_name(text)
    
    final_results.append({
        "image": img_path,
        "predicted_name": matched_name,
        "score": score
    })


In [136]:
from rapidfuzz import process
import re

# نبني database ذكية
def build_db(names):
    db = []
    
    for n in names:
        # الاسم كله
        db.append(n)
        
        # اسم الدوا بس
        base = re.sub(r'“.*?”', '', n).strip()
        db.append(base)
        
        # البراند
        brand = re.findall(r'“(.*?)”', n)
        if brand:
            db.append(brand[0])
    
    return list(set(db))


search_db = build_db(names_db)


# matching جديد
def smart_match(text):
    words = re.findall(r'[A-Za-z]+', text)
    
    best = None
    best_score = 0
    
    for w in words:
        match, score, _ = process.extractOne(w, search_db)
        
        if score > best_score:
            best = match
            best_score = score
    
    return best, best_score

In [ ]:


final_results = []

for item in data:
    img_path = item["image"]
    
    text = extract_text(img_path)
    text = clean_text(text)
    
    name, score = smart_match(text)

    
    final_results.append({
        "image": img_path,
        "predicted_name": name,
        "score": score
    })
print(final_results)


In [ ]:

# الفايل اللي فيه النتايج
data = final_results   # نفس اللي طلعته (image + predicted_name)

# فولدر الصور
images_folder = "images"

def clean_filename(name):
    # شيل أي رموز مش مسموح بيها في اسم الملف
    name = re.sub(r'[\\/*?:"<>|]', '', name)
    
    # شيل quotes
    name = name.replace("“", "").replace("”", "")
    
    # شيل مسافات زيادة
    name = re.sub(r"\s+", " ", name).strip()
    
    return name


used_names = set()

for item in data:
    old_path = item["image"]
    name = item["predicted_name"]
    
    if name is None:
        continue
    
    clean_name = clean_filename(name)
    
    # عشان لو فيه أسماء مكررة
    new_name = clean_name
    counter = 1
    
    while new_name in used_names:
        new_name = f"{clean_name}_{counter}"
        counter += 1
    
    used_names.add(new_name)
    
    new_path = os.path.join(images_folder, new_name + ".png")
    
    try:
        os.rename(old_path, new_path)
    except Exception as e:
        print(f"Error renaming {old_path}: {e}")

print("Done ✅")




Done ✅


In [156]:
import os
import re

folder = r"D:\VsCode Projects\pharmacy project\New folder"

seen = set()

for filename in os.listdir(folder):
    path = os.path.join(folder, filename)
    
    # شيل _1 _2 _3 من الاسم
    base = re.sub(r'_\d+', '', filename)
    
    if base in seen:
        os.remove(path)
    else:
        seen.add(base)

print("Removed duplicate names ✅")

Removed duplicate names ✅


In [147]:
import os

folder = r"D:\VsCode Projects\pharmacy project\images"

images = [f for f in os.listdir(folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

for img in images:
    print(img)
print(f"Total images: {len(images)}")

Abilia.png
Abilify 10-15-5.png
Acitretin 10-25.png
Actos 15-30.png
Acyclovir 400-800.png
Adancor 10-20.png
Adwitine 30-60.png
Agovald 25.png
anarcol.png
Anastrazol Sandoz.png
Andorivaban 2.5-10-15-20.png
Andovimpamid e 50-100-200.png
Angiobradine 5-7.5.png
Anselacox 60-90.png
Anvizram 200.png
Apetoid 20.png
Apexidon 0.5-1-2-3-4.png
Apixatrack 2.5-5.png
Arcoxia 60-90.png
Aricept 5-10.png
Arimidex 1.png
Aripiprex 10-30-vial.png
Aromasin 25.png
Arthfree 20.png
Artixiban 2.5-5.png
Atomafutix 10-25-40.png
Atomox apex 10-18-25-40-60.png
Atoreza 10l10.png
Atozemb 10l10.png
Atozet 10l10.png
Attensera 18.png
Avara 10-20.png
Avipravir 200.png
Avodart 0.5.png
Betmiga 50.png
Bladogra 50.png
Bradipect 5-7.5.png
Brilique 60-90.png
Brintellix 10-20.png
Budexan inh 1-0.5.png
Carcemia 100.png
Cardura 1-4-dxl.png
Carfalone 25-50.png
Casodex 50.png
Celebrex 200-100.png
Cellcept 250- 500.png
Champix 1.png
Cialis 5-20.png
Cipla tropium 18 mcg.png
Cipralex10.png
Cipram 20.png
Cludine tech 0.5-1.png
Concor a

In [157]:
brand_to_active = {
    "CYMBALTA": "Duloxetine",
    "ARIMIDEX": "Anastrozole",
    "LACOVIMP": "Lacosamide",
    "XARELTO": "Rivaroxaban",
    "ELIQUIS": "Apixaban",
    "PROCORALAN": "Ivabradine",
    "ARCOXIA": "Etoricoxib",
    "STRATTERA": "Atomoxetine",
    "ATOZET": "Atorvastatin + Ezetimibe",
    "BETMIGA": "Mirabegron",
    "SPIRIVA": "Tiotropium",
    "RESOLOR": "Prucalopride",
    "TECAVIR": "Entecavir",
    "JARDIANCE": "Empagliflozin",
    "SYNJARDY": "Empagliflozin + Metformin",
    "XIGDUO": "Dapagliflozin + Metformin",
    "VFEND": "Voriconazole",
    "FEMARA": "Letrozole",
    "ZOFRAN": "Ondansetron",
    "ZYPREXA": "Olanzapine",
    "TAMIFLU": "Oseltamivir",
    "PLAVIX": "Clopidogrel",
    "BRILIQUE": "Ticagrelor",
    "EBIXA": "Memantine",
    "ROACCUTAN": "Isotretinoin",
    "SEROQUEL": "Quetiapine",
}


def extract_info(line):
    # extract brand
    brand_match = re.findall(r'“(.*?)”', line)
    
    brand = brand_match[0].upper().strip() if brand_match else None
    
    # clean name (remove brand)
    name = re.sub(r'“.*?”', '', line).strip()
    
    # get active ingredient
    active = brand_to_active.get(brand, None)
    
    return {
        "name": name,
        "brand": brand,
        "active": active
    }

In [158]:
results = []

for line in names:   # lines = القائمة اللي عندك
    info = extract_info(line)
    results.append(info)

print(results[:10])

[{'name': 'Abilify 10-15-5', 'brand': None, 'active': None}, {'name': 'Abilia -15-20', 'brand': 'ABILIFY', 'active': None}, {'name': 'Acitretin 10-25', 'brand': None, 'active': None}, {'name': 'Actos 15-30', 'brand': None, 'active': None}, {'name': 'Acyclovir 400-800', 'brand': None, 'active': None}, {'name': 'Adancor 10-20', 'brand': None, 'active': None}, {'name': 'Adwitine 30-60', 'brand': 'CYMBALTA', 'active': 'Duloxetine'}, {'name': 'Agovald 25', 'brand': None, 'active': None}, {'name': 'anarcol', 'brand': None, 'active': None}, {'name': 'Anastrazol Sandoz', 'brand': 'ARIMIDEX', 'active': 'Anastrozole'}]


In [159]:

for item in results:   # القائمة اللي عندك
    name = item["name"]
    brand = item["brand"]
    
    if brand:
        full_name = f"{name} {brand}"
    else:
        full_name = name
    
    new_names.append(full_name)

# عرض النتيجة
print(new_names)

['Abilia -15-20 ABILIFY', 'Abilify 10-15-5', 'Acitretin 10-25', 'Actos 15-30', 'Acyclovir 400-800', 'Adancor 10-20', 'Adwitine 30-60 CYMBALTA', 'Agovald 25', 'Anastrazol Sandoz ARIMIDEX', 'Anastrazol synthon ARIMIDEX', 'Andorivaban 2.5-10-15-20 XARELTO', 'Andovimpamid e 50-100-200 LACOVIMP', 'Angiobradine 5-7.5 PROCORALAN', 'Anselacox 60-90 ARCOXIA', 'Anvizram 200', 'Apetoid 20 AVARA', 'Apexidon 0.5-1-2-3-4 Syrup RISPERDAL', 'Apixatrack 2.5-5 ELIQUIS', 'Arcoxia 60-90', 'Aricept 5-10', 'Arimidex 1', 'Aripiprex 10-30-vial ABILIFY', 'Aromasin 25', 'Arthfree 20 AVARA', 'Artixiban 2.5-5 ELIQUIS', 'Atomafutix 10-25-40 STRATTERA', 'Atomox apex 10-18-25-40-60 STRATTERA', 'Atoreza 10l10 20l10 40l10 “atozet “', 'Atozemb 10l10 20l10 “atozet “', 'Atozet 10l10 20l10 40l10', 'Attensera 18 25 40 STRATTERA', 'Avara 10-20', 'Avipravir 200', 'Avodart 0.5', 'Betmiga 50', 'Bladogra 50 BETMIGA', 'Bradipect 5-7.5 PROCORALAN', 'Brilique 60-90', 'Brintellix 10-20', 'Budexan inh 1-0.5', 'Carcemia 100 400', 'Ca

In [162]:
import os
from rapidfuzz import process

folder = r"D:\VsCode Projects\pharmacy project\New folder"

# خد snapshot للأسماء قبل أي تعديل
files = [f for f in os.listdir(folder) if f.endswith(".png")]

  # القائمة بتاعتك

# نحفظ اللي اتغير عشان مانكررش
used = set()

for file in files:
    old_path = os.path.join(folder, file)
    
    # تأكد إن الملف لسه موجود
    if not os.path.exists(old_path):
        continue
    
    name = file.replace(".png", "")
    
    match, score, _ = process.extractOne(name, new_names)
    
    if score > 80 and match not in used:
        
        clean_name = match.replace("/", "-").replace("\\", "-")
        clean_name = clean_name.replace(":", "").replace("*", "")
        
        new_path = os.path.join(folder, clean_name + ".png")
        
        try:
            os.rename(old_path, new_path)
            used.add(match)
            print(f"{file} → {clean_name}.png")
        except Exception as e:
            print(f"Error: {e}")

Abilia -15-20 ABILIFY.png → Abilia -15-20 ABILIFY.png
Abilify 10-15-5.png → Abilify 10-15-5.png
Acitretin 10-25.png → Acitretin 10-25.png
Actos 15-30.png → Actos 15-30.png
Acyclovir 400-800.png → Acyclovir 400-800.png
Adancor 10-20.png → Adancor 10-20.png
Adwitine 30-60 CYMBALTA.png → Adwitine 30-60 CYMBALTA.png
Agovald 25.png → Agovald 25.png
anarcol.png → anarcol.png
Anastrazol Sandoz ARIMIDEX.png → Anastrazol Sandoz ARIMIDEX.png
Andorivaban 2.5-10-15-20 XARELTO.png → Andorivaban 2.5-10-15-20 XARELTO.png
Andovimpamid e 50-100-200 LACOVIMP.png → Andovimpamid e 50-100-200 LACOVIMP.png
Angiobradine 5-7.5 PROCORALAN.png → Angiobradine 5-7.5 PROCORALAN.png
Anselacox 60-90 ARCOXIA.png → Anselacox 60-90 ARCOXIA.png
Anvizram 200.png → Anvizram 200.png
Apetoid 20 AVARA.png → Apetoid 20 AVARA.png
Apexidon 0.5-1-2-3-4 Syrup RISPERDAL.png → Apexidon 0.5-1-2-3-4 Syrup RISPERDAL.png
Apixatrack 2.5-5 ELIQUIS.png → Apixatrack 2.5-5 ELIQUIS.png
Arcoxia 60-90.png → Arcoxia 60-90.png
Aricept 5-10.png →

In [164]:
import json

with open("drug_names.json", "w", encoding="utf-8") as f:
    json.dump(new_names, f, ensure_ascii=False, indent=2)

print("Saved ✅")

Saved ✅


In [165]:
#read json
import json

with open("drug_names.json", "r", encoding="utf-8") as f:
    DRUG_DB = json.load(f)

# لو JSON عبارة عن list dict
if isinstance(DRUG_DB[0], dict):
    DRUG_DB = [d["name"] for d in DRUG_DB]

In [166]:
from rapidfuzz import process
import re

class DrugMatcher:
    def __init__(self, drug_list):
        self.drug_list = drug_list

    def clean_text(self, text):
        text = text.lower()
        text = re.sub(r'[^a-z0-9 ]', ' ', text)
        return text

    def match(self, ocr_text):
        clean = self.clean_text(ocr_text)

        match, score, _ = process.extractOne(
            clean,
            self.drug_list,
            scorer=process.fuzz.token_sort_ratio
        )

        return {
            "matched_name": match,
            "score": score
        }

# 1. image_processor



In [167]:
class MedicineOCRSystem:
    def __init__(self, engine='both', json_path="drug_names.json"):
        print("🚀 جاري تشغيل نظام OCR للأدوية...")
        
        self.ocr = OCREngine(engine=engine)
        self.extractor = MedicineDataExtractor()
        
        # 🔥 الجديد
        with open(json_path, "r", encoding="utf-8") as f:
            db = json.load(f)
            if isinstance(db[0], dict):
                db = [d["name"] for d in db]

        self.matcher = DrugMatcher(db)
        self.results = []

        print("✅ النظام جاهز!")

# 2.ocr_engine

In [168]:


class OCREngine:
    """محرك OCR يدعم Tesseract و EasyOCR"""

    def __init__(self, engine='both'):
        """
        engine: 'tesseract', 'easyocr', أو 'both'
        """
        self.engine = engine
        self.processor = ImageProcessor()

        if engine in ('easyocr', 'both'):
            print("⏳ جاري تحميل EasyOCR...")
            self.easy_reader = easyocr.Reader(
                ['en'],
                gpu=False  # حطها True لو عندك GPU
            )
            print("✅ EasyOCR جاهز!")

    def extract_text_tesseract(self, img):
        """استخراج النص باستخدام Tesseract"""
        # إعدادات Tesseract المحسنة للأدوية
        custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789./|mgMG '

        # معالجة الصورة
        processed = self.processor.preprocess(img)

        # استخراج النص
        text = pytesseract.image_to_string(processed, config=custom_config)

        # استخراج بيانات تفصيلية (مع confidence)
        details = pytesseract.image_to_data(
            processed,
            config=custom_config,
            output_type=pytesseract.Output.DICT
        )

        return {
            'text': text.strip(),
            'details': details,
            'engine': 'tesseract'
        }

    def extract_text_easyocr(self, img):
        """استخراج النص باستخدام EasyOCR"""
        results = self.easy_reader.readtext(img)

        texts = []
        for (bbox, text, confidence) in results:
            texts.append({
                'text': text,
                'confidence': confidence,
                'bbox': bbox
            })

        full_text = ' '.join([t['text'] for t in texts])

        return {
            'text': full_text,
            'details': texts,
            'engine': 'easyocr'
        }

    def extract(self, image_path):
        """استخراج النص من صورة"""
        img = self.processor.load_image(image_path)
        img = self.processor.resize_image(img)
        img = self.processor.sharpen_image(img)

        results = {}

        if self.engine in ('tesseract', 'both'):
            results['tesseract'] = self.extract_text_tesseract(img)

        if self.engine in ('easyocr', 'both'):
            results['easyocr'] = self.extract_text_easyocr(img)

        return results

    def extract_with_regions(self, image_path):
        """استخراج النص مع تحديد مناطق النص"""
        img = self.processor.load_image(image_path)
        img = self.processor.resize_image(img)

        # اكتشاف مناطق النص
        regions = self.processor.detect_text_regions(img)

        region_results = []
        for (x, y, w, h) in regions:
            # قص منطقة النص
            roi = img[y:y+h, x:x+w]

            if self.engine in ('easyocr', 'both'):
                result = self.extract_text_easyocr(roi)
            else:
                result = self.extract_text_tesseract(roi)

            region_results.append({
                'region': (x, y, w, h),
                'result': result
            })

        return region_results

# 4.data_extractor

In [169]:
class MedicineOCRSystem:
    def __init__(self, engine='both', json_path="drug_names.json"):
        print("🚀 جاري تشغيل نظام OCR للأدوية...")
        
        self.ocr = OCREngine(engine=engine)
        self.extractor = MedicineDataExtractor()
        
        # 🔥 الجديد
        with open(json_path, "r", encoding="utf-8") as f:
            db = json.load(f)
            if isinstance(db[0], dict):
                db = [d["name"] for d in db]

        self.matcher = DrugMatcher(db)
        self.results = []

        print("✅ النظام جاهز!")

In [ ]:
def process_single_image(self, image_path):
    print(f"\n📷 جاري معالجة: {image_path}")

    ocr_results = self.ocr.extract(image_path)

    combined_text = ""
    if 'easyocr' in ocr_results:
        combined_text += ocr_results['easyocr']['text'] + " "
    if 'tesseract' in ocr_results:
        combined_text += ocr_results['tesseract']['text']

    print(f"📝 OCR: {combined_text[:100]}...")

    # 🧠 match مع JSON
    match_result = self.matcher.match(combined_text)

    print(f"🎯 أقرب دوا: {match_result['matched_name']}")
    print(f"📊 Score: {match_result['score']}")

    return {
        "image": image_path,
        "ocr_text": combined_text,
        "matched_drug": match_result['matched_name'],
        "confidence": match_result['score']
    }

🚀 جاري تشغيل نظام OCR للأدوية...

Using CPU. Note: This module is much faster with a GPU.



⏳ جاري تحميل EasyOCR...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete✅ EasyOCR جاهز!
✅ النظام جاهز!

📁 جاري معالجة المجلد: images/


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'images/'

In [170]:
def process_single_image(self, image_path):
    print(f"\n📷 جاري معالجة: {image_path}")

    ocr_results = self.ocr.extract(image_path)

    combined_text = ""
    if 'easyocr' in ocr_results:
        combined_text += ocr_results['easyocr']['text'] + " "
    if 'tesseract' in ocr_results:
        combined_text += ocr_results['tesseract']['text']

    print(f"📝 OCR: {combined_text[:100]}...")

    # 🧠 match مع JSON
    match_result = self.matcher.match(combined_text)

    print(f"🎯 أقرب دوا: {match_result['matched_name']}")
    print(f"📊 Score: {match_result['score']}")

    return {
        "image": image_path,
        "ocr_text": combined_text,
        "matched_drug": match_result['matched_name'],
        "confidence": match_result['score']
    }

In [172]:
# ==============================
#   OCR + MATCH WITH JSON (FULL CELL)
# ==============================

import os
import json
import re
import cv2
import numpy as np
import easyocr
from rapidfuzz import process, fuzz

# ==============================
# تحميل JSON
# ==============================
JSON_PATH = "drug_names.json"  # عدّل المسار

with open(JSON_PATH, "r", encoding="utf-8") as f:
    DRUG_DB = json.load(f)

if isinstance(DRUG_DB[0], dict):
    DRUG_DB = [d["name"] for d in DRUG_DB]

print(f"Loaded {len(DRUG_DB)} drugs ✅")

# ==============================
# OCR INIT
# ==============================
reader = easyocr.Reader(['en'], gpu=False)

# ==============================
# IMAGE PROCESSING
# ==============================
def preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    thresh = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )
    return thresh

# ==============================
# OCR
# ==============================
def extract_text(image_path):
    img = cv2.imread(image_path)
    img = preprocess(img)

    results = reader.readtext(img)
    text = " ".join([r[1] for r in results])

    return text

# ==============================
# CLEAN TEXT
# ==============================
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ==============================
# MATCHING
# ==============================
def match_drug(ocr_text):
    clean = clean_text(ocr_text)

    match, score, _ = process.extractOne(
        clean,
        DRUG_DB,
        scorer=fuzz.token_sort_ratio
    )

    if score < 70:
        return None, score

    return match, score

# ==============================
# MAIN FUNCTION
# ==============================
def predict(image_path):
    print(f"\n📷 {image_path}")

    text = extract_text(image_path)
    print(f"📝 OCR: {text[:100]}...")

    drug, score = match_drug(text)

    print(f"🎯 Prediction: {drug}")
    print(f"📊 Confidence: {score}")

    return {
        "image": image_path,
        "ocr_text": text,
        "predicted_drug": drug,
        "confidence": score
    }

# ==============================
# TEST
# ==============================
# جرب على صورة واحدة
result = predict("Andorivaban 2.5-10-15-20 XARELTO.png")
print(result)

# ==============================
# BATCH (اختياري)
# ==============================
def process_folder(folder):
    results = []

    for file in os.listdir(folder):
        if file.lower().endswith(('.png','.jpg','.jpeg')):
            path = os.path.join(folder, file)
            res = predict(path)
            results.append(res)

    return results

# مثال:
# all_results = process_folder("images")

Using CPU. Note: This module is much faster with a GPU.


Loaded 654 drugs ✅

📷 Andorivaban 2.5-10-15-20 XARELTO.png


c:\Users\Abdelrahman\anaconda3\envs\New_test\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


📝 OCR: Andorivaban Rivaroiaban 10m9_ 1_ Imacy...
🎯 Prediction: None
📊 Confidence: 45.070422535211264
{'image': 'Andorivaban 2.5-10-15-20 XARELTO.png', 'ocr_text': 'Andorivaban Rivaroiaban 10m9_ 1_ Imacy', 'predicted_drug': None, 'confidence': 45.070422535211264}
